# LaplacianNB Basic Usage Tutorial

This notebook demonstrates the basic usage of LaplacianNB with molecular fingerprints, following the pattern from the original bayes_tutorial but showcasing both implementations.

## Package Installation and Imports

First, let's install the package and import necessary libraries.

In [ ]:
# Install the package (uncomment if needed)
# !pip install laplaciannb --upgrade

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# Import both implementations
from laplaciannb.LaplacianNB import LaplacianNB as LaplacianNB_Original
from laplaciannb.LaplacianNB_new import LaplacianNB as LaplacianNB_New
from laplaciannb.fingerprint_utils import convert_fingerprints

## Utility Function for Molecular Fingerprints

We'll create a memory-efficient function to calculate Morgan fingerprints from SMILES.

In [ ]:
def get_fp(smiles: str, n_bits: int = 1024) -> set:
    """
    Calculate Morgan fingerprint from SMILES string.
    
    Args:
        smiles (str): SMILES string
        n_bits (int): Size of folded fingerprint (default: 1024)
        
    Returns:
        set: Set of indices where bits are set to 1
    """
    mol = Chem.MolFromSmiles(smiles)
    
    if not mol:
        return set()
    
    # Use folded fingerprint for memory efficiency
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    fp = mfpgen.GetFingerprint(mol)
    
    if not fp:
        return set()
    
    return set(fp.GetOnBits())

## Create Example Dataset

Let's create a dataset with various molecules and their activities.

In [ ]:
# Create example DataFrame with diverse molecules
df = pd.DataFrame({
    "smiles": [
        "N[C@]([H])(C)C(=O)O",           # Alanine (amino acid)
        "O=Cc1ccc(O)c(OC)c1",            # Vanillin (aromatic aldehyde)
        "CN=C=O",                         # Methyl isocyanate
        "CCO",                            # Ethanol (alcohol)
        "c1ccccc1",                       # Benzene (aromatic)
        "CC(=O)O",                        # Acetic acid
        "CCCCO",                          # Butanol (alcohol)
        "c1ccc(C)cc1",                    # Toluene (aromatic)
    ],
    "activity": [1, 0, 0, 1, 0, 1, 1, 0],
})

In [ ]:
# Display the dataset
df

## Calculate Molecular Fingerprints

Convert SMILES to molecular fingerprints using our utility function.

In [ ]:
# Calculate fingerprints for each molecule
print("Calculating molecular fingerprints...")
df["fingerprints"] = df["smiles"].apply(lambda x: get_fp(x, n_bits=1024))

In [ ]:
# Display fingerprint information
print("Dataset with fingerprints:")
for idx, row in df.iterrows():
    fp_size = len(row["fingerprints"])
    fp_preview = list(sorted(row["fingerprints"]))[:5] if row["fingerprints"] else []
    print(f"  {row['smiles'][:25]:25} -> {fp_size:3d} bits, first 5: {fp_preview}")

In [ ]:
# Show the complete dataframe
df

## Prepare Training Data

Extract features (X) and targets (y) from our dataset.

In [ ]:
# Prepare data for training
X = df["fingerprints"].values
y = df["activity"].values

print(f"Training data shape: {X.shape}")
print(f"Target distribution: {np.bincount(y)}")
print(f"Classes: {np.unique(y)}")

## Example 1: Original LaplacianNB Implementation

Let's use the original LaplacianNB implementation that works with sets.

In [ ]:
# Create and train original classifier
clf_original = LaplacianNB_Original()
clf_original.fit(X, y)

### Get Joint Log-Likelihood

This shows the sum of feature probabilities for each compound per class.

In [ ]:
# Get joint log-likelihood (internal method)
joint_ll = clf_original._joint_log_likelihood(X)
print("Joint log-likelihood shape:", joint_ll.shape)
joint_ll

### Get Class Probabilities

Get probability predictions for each class using sklearn-compatible interface.

In [ ]:
# Get probability predictions
probabilities = clf_original.predict_proba(X)
print("Probabilities shape:", probabilities.shape)
probabilities

### Get Class Predictions

Get hard predictions for each sample.

In [ ]:
# Get class predictions
predictions = clf_original.predict(X)
print("Predictions:", predictions)
predictions

### Explore Model Properties

Let's examine the trained model's properties.

In [ ]:
# Get class names
print("Classes:", clf_original.classes_)
clf_original.classes_

In [ ]:
# Get feature mapping (index -> feature space position)
print("Number of unique features:", len(clf_original.feature_names_))
print("First 10 feature mappings:", dict(list(clf_original.feature_names_.items())[:10]))

In [ ]:
# Get feature log probabilities
print("Feature log probabilities shape:", clf_original.feature_log_prob_.shape)
print("Feature log probabilities (first 5 features):")
clf_original.feature_log_prob_[:, :5]

## Example 2: New sklearn-compatible LaplacianNB

Now let's use the new implementation that works with sklearn sparse matrices.

In [ ]:
# Convert fingerprints to sklearn format (sparse CSR by default)
X_sklearn = convert_fingerprints(X, n_bits=1024)
print(f"Sklearn format shape: {X_sklearn.shape}")
print(f"Sparse matrix format: {X_sklearn.format}")
print(f"Number of non-zero elements: {X_sklearn.nnz}")
print(f"Sparsity: {1 - X_sklearn.nnz / (X_sklearn.shape[0] * X_sklearn.shape[1]):.3f}")

In [ ]:
# Create and train new classifier
clf_new = LaplacianNB_New()
clf_new.fit(X_sklearn, y)

In [ ]:
# Get predictions with new implementation
predictions_new = clf_new.predict(X_sklearn)
print("Predictions (new):", predictions_new)
predictions_new

In [ ]:
# Get probabilities with new implementation
probabilities_new = clf_new.predict_proba(X_sklearn)
print("Probabilities shape:", probabilities_new.shape)
probabilities_new

In [ ]:
# Get log probabilities (additional method in new implementation)
log_probabilities_new = clf_new.predict_log_proba(X_sklearn)
print("Log probabilities shape:", log_probabilities_new.shape)
log_probabilities_new

### New Implementation Properties

In [ ]:
print("Classes:", clf_new.classes_)
print("Number of features:", clf_new.n_features_in_)
print("Feature count shape:", clf_new.feature_count_.shape)
print("Feature log probabilities shape:", clf_new.feature_log_prob_.shape)

## Example 3: Implementation Comparison

Let's compare the results from both implementations.

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'SMILES': df['smiles'],
    'True_Activity': y,
    'Original_Pred': predictions,
    'New_Pred': predictions_new,
    'Original_Prob_0': probabilities[:, 0],
    'Original_Prob_1': probabilities[:, 1],
    'New_Prob_0': probabilities_new[:, 0],
    'New_Prob_1': probabilities_new[:, 1],
})

comparison_df

In [ ]:
# Check if predictions match
predictions_match = np.array_equal(predictions, predictions_new)
probabilities_match = np.allclose(probabilities, probabilities_new, atol=1e-6)

print(f"Predictions match: {predictions_match}")
print(f"Probabilities match (within 1e-6): {probabilities_match}")

if not probabilities_match:
    prob_diff = np.abs(probabilities - probabilities_new)
    max_diff = np.max(prob_diff)
    mean_diff = np.mean(prob_diff)
    print(f"Maximum probability difference: {max_diff:.2e}")
    print(f"Mean probability difference: {mean_diff:.2e}")

## Example 4: Different Fingerprint Sizes

Let's explore how different fingerprint sizes affect performance.

In [ ]:
# Test different fingerprint sizes
fingerprint_sizes = [256, 512, 1024, 2048]
results = []

for n_bits in fingerprint_sizes:
    # Calculate fingerprints with current size
    fps = df["smiles"].apply(lambda x: get_fp(x, n_bits=n_bits)).values
    X_sized = convert_fingerprints(fps, n_bits=n_bits)
    
    # Train classifier
    clf_sized = LaplacianNB_New()
    clf_sized.fit(X_sized, y)
    
    # Calculate metrics
    accuracy = clf_sized.score(X_sized, y)
    sparsity = 1 - X_sized.nnz / (X_sized.shape[0] * X_sized.shape[1])
    avg_bits_per_molecule = np.mean([len(fp) for fp in fps])
    
    results.append({
        'n_bits': n_bits,
        'accuracy': accuracy,
        'sparsity': sparsity,
        'avg_bits_per_mol': avg_bits_per_molecule,
        'total_features': X_sized.shape[1]
    })

# Display results
results_df = pd.DataFrame(results)
results_df

## Example 5: Detailed Prediction Analysis

Let's analyze individual predictions in detail.

In [ ]:
# Detailed analysis for each molecule
print("Detailed Prediction Analysis:")
print("=" * 80)

for i, row in df.iterrows():
    smiles = row['smiles']
    true_activity = y[i]
    pred_orig = predictions[i]
    pred_new = predictions_new[i]
    prob_orig = probabilities[i]
    prob_new = probabilities_new[i]
    
    print(f"\nMolecule {i+1}: {smiles}")
    print(f"  True activity: {true_activity}")
    print(f"  Original prediction: {pred_orig} (prob: [{prob_orig[0]:.3f}, {prob_orig[1]:.3f}])")
    print(f"  New prediction: {pred_new} (prob: [{prob_new[0]:.3f}, {prob_new[1]:.3f}])")
    
    if pred_orig != true_activity:
        print(f"  ⚠️  Original implementation misclassified")
    if pred_new != true_activity:
        print(f"  ⚠️  New implementation misclassified")
    if pred_orig == pred_new == true_activity:
        print(f"  ✅ Both implementations correct")

## Summary

This tutorial demonstrated:

1. **Basic usage** of both LaplacianNB implementations
2. **Fingerprint calculation** with memory-efficient folded fingerprints
3. **Model training and prediction** with molecular data
4. **Implementation comparison** showing compatibility between versions
5. **Fingerprint size optimization** for different use cases
6. **Detailed analysis** of individual predictions

### Key Takeaways:

- Both implementations produce identical results
- The new implementation is sklearn-compatible and memory-efficient
- Fingerprint size affects sparsity and potentially accuracy
- The package handles molecular fingerprints effectively for classification tasks